In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold, train_test_split

from my_project.data.utils import load_column_config

np.random.seed(42)

In [ ]:
load_dotenv(override=True)

CONFIGS_DIR = os.getenv("CONFIGS_DIR")
DATA_DIR = os.getenv("DATA_DIR")
RESULTS_DIR = os.getenv("RESULTS_DIR")

In [ ]:
with open(os.path.join(CONFIGS_DIR, "data.yaml")) as f:
    data_config = yaml.safe_load(f)

col_cfg = load_column_config(CONFIGS_DIR)

In [ ]:
df = pd.read_pickle(
    os.path.join(DATA_DIR, data_config["paths"]["preprocessed_data_path"])
)

In [ ]:
split_pct = data_config["preprocessing"]["split"]
target_col = col_cfg.TARGET

In [ ]:
df_train_valid, df_test = train_test_split(
    df, test_size=split_pct["test"], random_state=42, stratify=df[target_col]
)

df_train, df_valid = train_test_split(
    df_train_valid,
    test_size=split_pct["valid"] / (split_pct["train"] + split_pct["valid"]),
    random_state=42,
    stratify=df_train_valid[target_col],
)

print(f"Train set size: {len(df_train)}")
print(f"Validation set size: {len(df_valid)}")
print(f"Test set size: {len(df_test)}")

In [ ]:
split_index_dict = {
    "train": df_train.index,
    "valid": df_valid.index,
    "test": df_test.index,
    "train_valid": df_train_valid.index,
}

In [ ]:
with open(
    os.path.join(DATA_DIR, data_config["paths"]["split_indices_path"]), "wb"
) as f:
    pd.to_pickle(split_index_dict, f)

# Cross validation split

In [ ]:
crossval_split = StratifiedKFold(**data_config["preprocessing"]["cv"])

crossval_indices = [
    {
        "train": df_train_valid.index[train_index],
        "valid": df_train_valid.index[valid_index],
        "test": df_test.index,
    }
    for train_index, valid_index in list(
        crossval_split.split(df_train_valid, df_train_valid[target_col])
    )
]

In [ ]:
with open(
    os.path.join(DATA_DIR, data_config["paths"]["crossval_indices_path"]), "rb"
) as f:
    pickle.load(f)

# with open(
#     os.path.join(DATA_DIR, data_config["paths"]["crossval_indices_path"]), "wb"
# ) as f:
#     pickle.dump(crossval_indices, f)

In [ ]:
# Additional CV splits: train-only subsamples that are strict subsets of original folds.
SUBSAMPLE_PCTS = [0.5, 0.25, 0.1, 0.05, 0.01]
SUBSAMPLE_SEEDS = [12, 123, 1234]
SUBSAMPLE_SEEDS = [12]


crossval_subsampled_indices = {}

for p in SUBSAMPLE_PCTS:
    for seed in SUBSAMPLE_SEEDS:
        key = f"p{int(p * 100)}_seed{seed}"
        folds_subsampled = []

        for fold in crossval_indices:
            train_idx = np.array(fold["train"])
            y_train_fold = df.loc[train_idx, target_col]

            sample_size = max(1, int(np.floor(len(train_idx) * p)))
            if sample_size >= len(train_idx):
                train_sub_idx = train_idx.copy()
            else:
                train_sub_idx, _ = train_test_split(
                    train_idx,
                    train_size=sample_size,
                    random_state=seed,
                    stratify=y_train_fold,
                )

            train_sub_idx = pd.Index(train_sub_idx).sort_values()
            assert set(train_sub_idx).issubset(set(train_idx))

            folds_subsampled.append(
                {
                    "train": train_sub_idx,
                    "valid": fold["valid"],
                    "test": fold["test"],
                }
            )

        crossval_subsampled_indices[key] = folds_subsampled

summary_rows = []
for key, folds_ in crossval_subsampled_indices.items():
    first_fold = folds_[0]
    summary_rows.append(
        {
            "split_key": key,
            "folds": len(folds_),
            "train_size_fold0": len(first_fold["train"]),
            "valid_size_fold0": len(first_fold["valid"]),
            "test_size_fold0": len(first_fold["test"]),
        }
    )

pd.DataFrame(summary_rows).sort_values("split_key").reset_index(drop=True)

In [ ]:
base_cv_path = os.path.join(DATA_DIR, data_config["paths"]["crossval_indices_path"])
base_cv_stem, base_cv_ext = os.path.splitext(base_cv_path)
crossval_subsampled_indices_path = f"{base_cv_stem}_subsampled{base_cv_ext}"

with open(crossval_subsampled_indices_path, "wb") as f:
    pickle.dump(crossval_subsampled_indices, f)

print("Saved:", crossval_subsampled_indices_path)
print("Keys:", sorted(crossval_subsampled_indices.keys()))